# Aula 2, Classificação

Notebook executável que acompanha a aula [02-classificacao.md](../../lessons/modulo-02-fundamentos-ml/02-classificacao.md).

## O que vamos fazer aqui

Agora a resposta não é um número, e sim uma categoria: o aluno será aprovado ou não.
Esse é um problema de classificação, e vamos resolvê-lo com uma regressão logística
implementada do zero, prevendo a aprovação a partir das horas de estudo e da
frequência às aulas.

O motor de otimização é o mesmo da regressão, o gradiente descendente. O que muda é a
função que produz a saída, a sigmoide, e a forma de medir o erro, a entropia cruzada.
O notebook usa apenas numpy.

## Dados de aprovação

Geramos dados em que alunos que estudam mais e frequentam mais tendem a ser
aprovados, com alguma sobreposição entre os grupos, como acontece na realidade. A
variável `aprovado` vale 1 ou 0.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n = 200
horas = rng.uniform(0, 10, size=n)
frequencia = rng.uniform(0, 1, size=n)
score = 0.6 * horas + 4 * frequencia - 4 + rng.normal(0, 1, size=n)
aprovado = (score > 0).astype(int)
X = np.column_stack([horas, frequencia])
print('aprovados:', int(aprovado.sum()), 'de', n)

Cada aluno é descrito por duas variáveis, empilhadas na matriz `X`. A
sobreposição entre aprovados e reprovados é proposital, pois é ela que impede uma
acurácia de 100% e torna o exemplo realista.

## Regressão logística do zero

A sigmoide transforma a combinação linear em uma probabilidade entre 0 e 1. O treino
ajusta os pesos por gradiente descendente, e as derivadas da entropia cruzada têm a
mesma forma simples que vimos na regressão linear.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def treinar_logistica(X, y, taxa=0.1, iteracoes=5000):
    """Ajusta os pesos por gradiente descendente, minimizando a entropia cruzada."""
    m, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    for _ in range(iteracoes):
        erro = sigmoide(X @ w + b) - y
        w -= taxa * (X.T @ erro) / m
        b -= taxa * np.mean(erro)
    return w, b


w, b = treinar_logistica(X, aprovado)
prob = sigmoide(X @ w + b)
previsto = (prob >= 0.5).astype(int)
acuracia = np.mean(previsto == aprovado)
print(f'Pesos: {np.round(w, 3)}, viés: {b:.3f}')
print(f'Acurácia no treino: {acuracia:.2%}')

novo = np.array([7, 0.8])
print(f'Probabilidade de aprovação (7h, freq 0.8): {sigmoide(novo @ w + b):.2%}')

A acurácia fica alta, mas não perfeita, justamente por causa da
sobreposição. Repare que o modelo devolve uma probabilidade, e não só um rótulo, o
que é valioso em educação, pois permite distinguir um aluno no limite da aprovação de
outro com folga.

## Projeto da aula, matriz de confusão

A acurácia sozinha esconde onde o modelo erra. A matriz de confusão abre os acertos e
erros por classe, mostrando quantos aprovados e reprovados foram classificados certo
e errado.

In [ ]:
# Conta acertos e erros por classe.
vp = int(np.sum((previsto == 1) & (aprovado == 1)))  # aprovado certo
vn = int(np.sum((previsto == 0) & (aprovado == 0)))  # reprovado certo
fp = int(np.sum((previsto == 1) & (aprovado == 0)))  # disse aprovado, era reprovado
fn = int(np.sum((previsto == 0) & (aprovado == 1)))  # disse reprovado, era aprovado

print('Matriz de confusão')
print(f'                 previu aprovado   previu reprovado')
print(f'era aprovado          {vp:4d}              {fn:4d}')
print(f'era reprovado         {fp:4d}              {vn:4d}')
print()
print('Onde o modelo mais erra costuma ser perto da fronteira de decisão.')

Os erros se concentram nos alunos perto da fronteira de decisão, onde a
probabilidade fica perto de 0,5. Para o projeto, observe em qual canto da matriz
estão os erros e relacione com esses casos limítrofes. Esse olhar para os erros
prepara a aula de validação.